## Week 2 - Day 3 - Linear Algebra for ML

BinX Tech AI & ML Internship

Topics:
- Why linear algebra is the language of ML
- Vectors: one sample's features
- Matrices: a full dataset (samples x features)
- The dot product and how models predict with it
- Matrix multiplication and the shape-matching rule

The parts under here are for testing and showcasing the lesson content, and the Day 3 task will be at the end.

## Vectors

A vector is an ordered list of numbers - in ML, usually one data sample's features.

In [1]:
import numpy as np

v = np.array([25, 50000, 3])  # a customer: age, income, tenure
print(v)
print("shape:", v.shape)

[   25 50000     3]
shape: (3,)


## Matrices

A matrix is a 2D grid of numbers - a full dataset, where each row is a sample and each column is a feature.
Shape = (rows, columns) = (samples, features).

In [2]:
X = np.array([[25, 50000, 3],
              [40, 80000, 10],
              [33, 62000, 5]])
print(X)
print("shape:", X.shape)  # (3, 3): 3 samples, 3 features

[[   25 50000     3]
 [   40 80000    10]
 [   33 62000     5]]
shape: (3, 3)


## The Dot Product

Multiplies two vectors element-by-element and sums the result into a single number. This is exactly how a linear
model predicts: prediction = dot(features, weights) + bias.

In [3]:
features = np.array([25, 50000, 3])
weights = np.array([0.1, 0.0002, 1.5])

prediction = np.dot(features, weights)
print(prediction)  # 2.5 + 10 + 4.5 = 17.0

17.0


## Matrix Multiplication

Applies the dot product across a whole matrix at once - predicting for every sample in a single operation.
Rule: an (m x n) matrix times an (n x p) matrix/vector gives an (m x p) result - the inner dimensions must match.

In [4]:
X_small = np.array([[25, 50000, 3], [40, 80000, 10]])  # (2, 3)
w_small = np.array([0.1, 0.0002, 1.5])                  # (3,)

predictions = X_small @ w_small
print(predictions)  # (2,): one prediction per sample

[17. 35.]


## Day 3 Tasks

hands-on lab, using three movies from the IMDb Top 1000 dataset as our
data samples.

## Step 1 - Represent Three Samples as a Matrix

Pick three movies and use `IMDB_Rating`, `Meta_score`, and `Runtime` (converted to minutes) as their features,
producing a (3 x 3) matrix - 3 samples, 3 features.

In [5]:
import pandas as pd

imdb = pd.read_csv("../imdb_top_1000.csv")

sample_movies = imdb[["Series_Title", "IMDB_Rating", "Meta_score", "Runtime"]].head(3).copy()
sample_movies["Runtime"] = sample_movies["Runtime"].str.replace(" min", "").astype(float)

print(sample_movies)

X = sample_movies[["IMDB_Rating", "Meta_score", "Runtime"]].to_numpy()
print("\nX:\n", X)
print("shape:", X.shape)  # (3, 3): 3 movies, 3 features

               Series_Title  IMDB_Rating  Meta_score  Runtime
0  The Shawshank Redemption          9.3        80.0    142.0
1             The Godfather          9.2       100.0    175.0
2           The Dark Knight          9.0        84.0    152.0

X:
 [[  9.3  80.  142. ]
 [  9.2 100.  175. ]
 [  9.   84.  152. ]]
shape: (3, 3)


## Step 2 - Dot Product by Hand, Then Verify

Give each feature an arbitrary weight and compute the "score" for the first movie by hand, then confirm it with
`np.dot`.

In [6]:
weights = np.array([10, 1, 0.5])  # made-up weights: rating matters most, then meta_score, then runtime

first_movie = X[0]

# By hand: (rating * 10) + (meta_score * 1) + (runtime * 0.5)
by_hand = (first_movie[0] * 10) + (first_movie[1] * 1) + (first_movie[2] * 0.5)
verified = np.dot(first_movie, weights)

print("By hand:", by_hand)
print("np.dot: ", verified)
print("Match:", by_hand == verified)

By hand: 244.0
np.dot:  244.0
Match: True


## Step 3 - Matrix Multiplication for All Three Samples at Once

In [7]:
scores = X @ weights  # (3, 3) @ (3,) -> (3,): one score per movie

for title, score in zip(sample_movies["Series_Title"], scores):
    print(f"{title}: {score:.2f}")

The Shawshank Redemption: 244.00
The Godfather: 279.50
The Dark Knight: 250.00


## Step 4 - Deliberately Trigger a Shape Mismatch

In [8]:
bad_weights = np.array([10, 1, 0.5, 2])  # 4 weights for a 3-feature matrix - on purpose

try:
    X @ bad_weights
except ValueError as e:
    print("Error:", e)

Error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 3)


**Why this happens:** `X` has shape `(3, 3)` - 3 movies, 3 features - while `bad_weights` has shape `(4,)`. Matrix
multiplication requires the inner dimensions to match: the number of columns in `X` (3) must equal the number of
elements in the weight vector. Since 3 != 4, NumPy raises a `ValueError` about mismatched dimensions rather than
silently guessing what was meant.

**The fix:** make sure the weight vector has exactly one weight per feature/column - here that means going back to
a 3-element weight vector (one for `IMDB_Rating`, one for `Meta_score`, one for `Runtime`). In practice, this exact
error is what you'll see whenever a model's weight vector doesn't match the number of features it was trained on -
e.g. after adding or dropping a column without retraining.